### **Read file from volume**

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

cust_df = spark.read \
    .option("header","true")\
    .option("interSchema", "true") \
    .csv("/Volumes/workspace/raw/rawvolume/rawdata/cust.csv")

cust_df.display()

### **Load into Stage Table**

In [0]:
cust_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.scd.stage_customer")



In [0]:
%sql
SELECT * FROM workspace.scd.stage_customer;

**SCD Type 1 — Simple** 

Requirement

If customer already exists:

➡️ Update the customer information.

If customer doesn't exist:

➡️ Insert new customer.

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.scd.dimention_customer (
    customer_id INT,
    customer_name STRING,
    city STRING,
    email STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
)
USING DELTA
""")

In [0]:
from delta.tables import DeltaTable

stage_df = spark.table("workspace.scd.stage_customer")

target = DeltaTable.forName(spark, "workspace.scd.dimention_customer")

target.alias("t") \
    .merge(stage_df.alias("s"),
           "t.customer_id = s.customer_id"
           )\
        .whenMatchedUpdate( set = {"customer_name": "s.customer_name",
                                   "city": "s.city",
                                   "email": "s.email"})\
        .whenNotMatchedInsert(values = {
                "customer_id" : "s.customer_id",
                "customer_name": "s.customer_name",
                "city" : "s.city",
                "email": "s.email"
        })\
        .execute()
    


In [0]:
%sql
SELECT * FROM workspace.scd.dimention_customer ;

**SCD Type 2 Logic**

If customer is new:

➡️ Insert customer.

If customer exists and data changed:

➡️ Expire old record
➡️ Insert new record

In [0]:
from pyspark.sql.functions import *

stage_df = spark.table("workspace.scd.stage_customer")

target = DeltaTable.forName(spark, "workspace.scd.dimention_customer")

# Expire old record

target.alias("t") \
    .merge(
        stage_df.alias("s"),
        """
        t.customer_id = s.customer_id AND t.is_current = true
        and ( t.customer_name <> s.customer_name OR t.city <> s.city OR t.email <> s.email )
        """
    )\
        .whenMatchedUpdate(
            set = {
                "end_date": "current_date()",
                "is_current": "false"
            }
        )\
            .execute()

### **Then insert the new/current record:**

In [0]:
stage_df.createOrReplaceTempView("vw_stage_customer")

spark.sql("""
INSERT INTO workspace.scd.dimention_customer
SELECT
    customer_id,
    customer_name,
    city,
    email,
    current_date(),
    DATE('9999-12-31'),
    true
FROM workspace.scd.stage_customer s
WHERE NOT EXISTS (
    SELECT 1
    FROM workspace.scd.dimention_customer t
    WHERE t.customer_id = s.customer_id
      AND t.is_current = true
      AND t.customer_name = s.customer_name
      AND t.city = s.city
      AND t.email = s.email
)
""")

In [0]:
%sql
SELECT * FROM workspace.scd.dimention_customer;

**Example**

Initially:

customer_id	name	city	start_date	end_date	current
101	Rahul	Pune	2026-08-01	9999-12-31	true

New file says Rahul moved to Mumbai.

**After SCD Type 2**:

customer_id	name	city	start_date	end_date	current

101	Rahul	    Pune	    2026-08-01	2026-08-20	false

101	Rahul	    Mumbai	    2026-08-20	9999-12-31	true

✅ History is maintained.